## Ch7-03 — Parameter sweep

This notebook introduces numpy parameter sweeps over the sympy-bound energy model; after running it you can show how delivered energy varies with heater power and mark the design requirement boundary on the plot.


Notebook 01 established that `Q_fn(800.0, 120.0, 0.7)` returns 67200 J. This notebook sweeps heater power from 500 W to 1200 W at fixed duration (120 s) and efficiency (0.7) to show which power values deliver enough energy for the toaster's function. The matplotlib figure is the simulation evidence referenced by the judgment record in Chapter 8. See [Ch7-01 symbolic binding](01-calc-energy.ipynb) for the lambdify setup.


In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
    action def ApplyHeat {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        out energy : Real;
        first start;
        then action calculate {
            assign energy := DeliveredEnergy(power, duration, efficiency);
        }
        then done;
    }
    item def Start;
    item def Finish;
    item def Cancel;
    allocate ApplyHeat to HeatingSystem;
    requirement def HeatingReq {
        subject heater : Heater;
        require constraint { heater.power >= 600.0 }
    }
    requirement heating : HeatingReq;
    part efficient : Heater;
    part weak : Heater { attribute :>> power = 400.0; }
    abstract part def HeatingElement;
    part def ResistanceCoil :> HeatingElement {
        attribute resistance : Real default = 12.0;
    }
    part def PowerWire :> HeatingElement {
        attribute gauge : Real default = 14.0;
    }
    part def HeatingAssembly :> HeatingSystem {
        part coil : ResistanceCoil;
        part wire : PowerWire;
    }
    part heatingEvidence {
        assert satisfy heating by efficient;
        assert satisfy heating by weak;
    }
    part def BreadLoader { part bread : Start; }
    part def BreadEjector { part bread : Finish; }
    state Cycle {
        entry; then idle;
        state idle;
        state heating;
        state ready;
        state cancelled;
        transition first idle accept Start then heating;
        transition first heating accept Finish then ready;
        transition first heating accept Cancel then cancelled;
    }

    part def BreadHandling {
        part loader : BreadLoader;
        part ejector : BreadEjector;
        flow loader.bread to ejector.bread;
    }
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"
print(f"Model ok: {model.ok}")


In [ ]:
# A calc def without the ScalarValues import cannot resolve 'Real' and fails to parse.
bad_source = """
package P {
    calc def Broken {
        in x : Real;
        return : Real = x * x;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok, "Expected failure: Real is undefined without ScalarValues::* import"
print(f"Negative control ok: bad.ok={bad.ok}")


In [ ]:
import sympy as sp
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from toaster.simulate import sweep_1d

# Rebuild the sympy binding (self-contained per SA-2 / fresh kernel)
P, t, eta = sp.symbols('P t eta', positive=True)
Q_fn = sp.lambdify([P, t, eta], P * t * eta, 'numpy')

# Sweep power 500–1200 W; duration=120 s, efficiency=0.7
P_vals = np.linspace(500, 1200, 50)
Q_vals = sweep_1d(Q_fn, P_vals, t=120.0, eta=0.7)

# Design threshold: 50000 J ensures toast within the cycle time at typical efficiency
threshold = 50_000.0

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(P_vals, Q_vals / 1000, label="Delivered energy")
ax.axhline(threshold / 1000, color="red", linestyle="--", label=f"Threshold {threshold/1000:.0f} kJ")
ax.set_xlabel("Heater power (W)")
ax.set_ylabel("Delivered energy (kJ)")
ax.set_title("Energy vs. heater power (t=120 s, η=0.7)")
ax.legend()
fig.tight_layout()

out = "ch07_param_sweep.svg"
fig.savefig(out)
plt.close(fig)

crossing_idx = np.argmax(Q_vals >= threshold)
print(f"Threshold crossed at: {P_vals[crossing_idx]:.0f} W")
print(f"Nominal (800 W): {Q_vals[P_vals >= 800][0] / 1000:.1f} kJ")
print(f"Figure saved: {out}")
conn.close()


The requirement that delivered energy exceeds the design threshold (A-F) is evaluated across a power sweep using `sweep_1d` and lambdify (O-S); the matplotlib figure shows the energy curve and marks the threshold crossing (E).


Try the chapter exercise in `exercises/ch07/exercise.ipynb`: sweep brew duration (60–180 s) at fixed power and efficiency for the coffee maker model and find the minimum duration that meets a 40 kJ energy threshold.
